# SPArta — qwen2.5:1.5b local-extraction benchmark (Google Colab, T4 GPU)

Runs the 21-question `data/sample_questions.json` sample through SPArta's **production pipeline** against the full 687-document corpus (`data/evidence_full`), using **Ollama + qwen2.5:1.5b** as the local semantic-extraction/planning provider instead of Gemini.

**Before running:** `Runtime -> Change runtime type -> T4 GPU`. Ollama auto-detects CUDA and uses the GPU for qwen2.5:1.5b; the local embedding model (`all-MiniLM-L6-v2`) also auto-detects and uses CUDA if present. Nothing in SPArta's code needs to change for either.

This notebook does **not** modify SPArta's architecture — it's the exact same `jaw_ingest.submit_cli` / `scripts/evaluate.py` you'd run locally, just pointed at a local Ollama endpoint and executed on Colab's hardware.

## 1. Confirm the T4 GPU is attached
If this errors or shows no GPU, go to `Runtime -> Change runtime type -> T4 GPU` and rerun from the top.

In [ ]:
!nvidia-smi

## 2. Clone SPArta
This gets the code and the small sample files (`data/sample_questions.json`, `data/questions.json`) — but **not** `data/evidence_full` (108MB, not committed to git). You'll provide that in the next step.

In [ ]:
!git clone https://github.com/AniHub-N/SPARta.git
%cd SPARta

## 3. Get `data/evidence_full` onto Colab
`data/evidence_full` isn't in git, so it has to come from you. Pick **one** of the two cells below and run only that one.

**Option A — upload a zip** (zip your local `data/evidence_full/` folder as `evidence_full.zip` first, e.g. `cd data && zip -r evidence_full.zip evidence_full`).

**Option B — Google Drive** (if you've already copied `evidence_full/` into your Drive).

In [ ]:
# --- Option A: upload evidence_full.zip directly ---
from google.colab import files
uploaded = files.upload()  # pick evidence_full.zip when prompted
!unzip -q -o evidence_full.zip -d data/
!ls data/evidence_full | head -5

In [ ]:
# --- Option B: mount Google Drive instead ---
# from google.colab import drive
# drive.mount('/content/drive')
# !cp -r "/content/drive/MyDrive/<path-to>/evidence_full" data/evidence_full
# !ls data/evidence_full | head -5

In [ ]:
# Sanity check before spending any time on setup: fail loudly now, not 20 minutes in.
import os
assert os.path.exists("data/evidence_full/documents.jsonl"), "data/evidence_full is missing or incomplete - go back to step 3."
assert os.path.exists("data/sample_questions.json"), "data/sample_questions.json missing - the git clone may have failed."
print("data/evidence_full and data/sample_questions.json are both present.")

## 4. Install Python dependencies

In [ ]:
!pip install -q -e .

## 5. Install Ollama and pull qwen2.5:1.5b
The official install script auto-detects the T4's CUDA driver and installs the GPU-enabled build - no extra flags needed.

In [ ]:
!curl -fsSL https://ollama.com/install.sh | sh

In [ ]:
import subprocess, time, requests

# Ollama has to run as a background server process; a Colab cell can't just run
# `ollama serve` in the foreground and still let you run later cells.
server = subprocess.Popen(
    ["ollama", "serve"],
    stdout=open("/content/ollama_server.log", "w"),
    stderr=subprocess.STDOUT,
)

for _ in range(30):
    try:
        requests.get("http://localhost:11434/api/version", timeout=2)
        print("Ollama server is up.")
        break
    except requests.exceptions.ConnectionError:
        time.sleep(1)
else:
    raise RuntimeError("Ollama server did not come up - check /content/ollama_server.log")

In [ ]:
!ollama pull qwen2.5:1.5b

In [ ]:
# Warm the model up with one call, then check `ollama ps` - the PROCESSOR column
# should say "100% GPU" (or a GPU/CPU split) if the T4 is actually being used.
!ollama run qwen2.5:1.5b "Say OK." --verbose
!ollama ps

## 6. Run the 21-question benchmark through the real SPArta pipeline
Lazy/DISCOVER-driven, exactly as `jaw-submit` runs locally - just pointed at Ollama instead of Gemini via environment variables (no code changes). The retrieval index (DuckDB + Qdrant) is built fresh on this machine, embedding the full 63,995-fragment corpus once; the LLM response cache uses a dedicated directory so every entry in it is a real Ollama call, not a stale result from another provider.

In [ ]:
import os, subprocess, time

env = os.environ.copy()
env.update({
    "JAW_LLM_PROVIDER": "openai_compatible",
    "JAW_LLM_BASE_URL": "http://localhost:11434/v1",
    "JAW_LLM_API_KEY": "ollama",  # ignored by Ollama, just needs to be non-empty
    "JAW_LLM_MODEL": "qwen2.5:1.5b",
    "JAW_LLM_TIMEOUT": "180",
})

start = time.monotonic()
result = subprocess.run(
    [
        "python", "-m", "jaw_ingest.submit_cli",
        "--questions", "data/sample_questions.json",
        "--evidence-root", "data/evidence_full",
        "--output", "submission_qwen21.csv",
        "--cache-dir", ".cache/llm_qwen",
    ],
    env=env,
    capture_output=True,
    text=True,
)
elapsed = time.monotonic() - start

print(result.stdout[-8000:])
if result.returncode != 0:
    print("STDERR:\n", result.stderr[-4000:])
print(f"\n=== submit_cli exit code: {result.returncode} | total runtime: {elapsed/60:.1f} min ({elapsed:.0f}s) ===")

## 7. Score the submission with the official evaluator

In [ ]:
!python scripts/evaluate.py --submission submission_qwen21.csv --questions data/sample_questions.json --per-question

## 8. Summary: total Ollama calls + runtime
Every file in the dedicated `.cache/llm_qwen` directory is one distinct real Ollama call this run made (planning, extraction, and verification calls are all cached there; repeated identical prompts within the run reuse a cache entry instead of double-counting).

In [ ]:
import glob
n_calls = len(glob.glob(".cache/llm_qwen/*.json"))
print(f"Total distinct Ollama calls (cached responses): {n_calls}")
print(f"Total wall-clock runtime: {elapsed/60:.1f} minutes ({elapsed:.0f}s)")